<a href="https://colab.research.google.com/github/mteuslms/AI-ML-projects/blob/main/LLMs_and_Transformers/Transformer_Build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as func

In [ ]:
#Embedding Layer
class EmbeddingLayer(nn.Module):
  def __init__(self,vocab_size,d_model):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,d_model)

  def forward(self,x):
    return self.embedding(x)


In [ ]:
# Self Attention
class SelfAttention(nn.Module):
  def __init__(self,d_model):
    super().__init__()
    self.Wq = nn.Linear(d_model,d_model)
    self.Wk = nn.Linear(d_model,d_model)
    self.Wv = nn.Linear(d_model,d_model)
    self.d_model = d_model

  def forward(self,x):
    q = self.Wq(x)
    k = self.Wk(x)
    v = self.Wv(x)

    attention_scores = torch.matmul(q,k.transpose(-2,-1))/torch.sqrt(torch.tensor(self.d_model))
    attention_weight = func.softmax(attention_scores,dim=-1)
    return torch.matmul(attention_weight,v)

In [ ]:
#MLP, feedfoward layer
class MLP(nn.Module):
  def __init__(self,d_model):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(d_model,d_model*4),
        nn.ReLU(),
        nn.Linear(d_model*4,d_model)
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,d_model):
    super().__init__()
    self.attention = SelfAttention(d_model)
    self.mlp = MLP(d_model)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def forward(self, x):
    x = x + self.attention(self.ln1(x))
    x = x + self.mlp(self.ln2(x))
    return x

In [ ]:
class MiniBatchTransformer(nn.Module):
  def __init__(self,vocab_size,d_model):
    super().__init__()
    self.embedding = EmbeddingLayer(vocab_size,d_model)
    self.block = TransformerBlock(d_model)
    self.un_embedding = nn.Linear(d_model,vocab_size)

  def forward(self,x):
    x = self.embedding(x)
    x = self.block(x)
    return self.un_embedding(x)

In [ ]:
# create a vocabulary
words_to_idx= {
    'Hello':0,
    'There':1,
    'World':2,
    'This':3,
    'is':4,
    'a':5,
    'mini':6,
    'batch':7,
}

idx_to_words = {v: k for k, v in words_to_idx.items()}

print(idx_to_words)

{0: 'Hello', 1: 'There', 2: 'World', 3: 'This', 4: 'is', 5: 'a', 6: 'mini', 7: 'batch'}


In [ ]:
#training process
vocab_size = len(words_to_idx)
d_model = 16
model = MiniBatchTransformer(vocab_size, d_model)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

#training data
x = torch.tensor([[0,1], [3,4], [4,5], [5,6]])
y = torch.tensor([2,5,6,7])

for epoch in range(200):
  optimizer.zero_grad()
  logits = model(x) # shape 4 x 2 x 8
  if epoch == 199:
    print(logits.shape)
  loss = loss_fn(logits[:, -1, :],y)
  loss.backward()
  optimizer.step()

torch.Size([4, 2, 8])


In [ ]:
model.eval()
with torch.no_grad():
  logits = model(torch.tensor([[0,1]]))
  pred = torch.argmax(logits[:, -1, :], dim = -1)
  print(idx_to_words[pred.item()])

World
